# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliakhtar1010/search-ranking-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Growing content has a different structural profile

The paper reports that growing content is longer and younger on average than declining content. Growing pages average about 3,180 words compared with 2,311 for declining pages, while their average age is about 184 days compared with 230 days.

**Methodology question:** How was the growing/declining label constructed, and were the word-count and age comparisons evaluated while controlling for other factors such as content type, client, search visibility, or topic difficulty?

The sample sizes are large, so the observed differences are useful directional evidence. However, because the comparison is observational, I would not interpret longer or younger content as causing growth by itself.

### Finding 2 — Refreshed mature content shows stronger measured performance

The paper reports that older content refreshed within 30 days had substantially stronger measured health and impressions than older stale content, including a reported 3.2× health difference and 57× impression difference in the analyzed portfolio.

**Methodology question:** Were refreshed and unrefreshed pages comparable before the refresh, or could the refreshed group have been selected because those pages already had stronger historical demand, visibility, or business importance?

The finding is useful for prioritizing refresh investigation, but the observational design does not establish that the refresh itself caused the full measured difference.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Random Split vs Client-Grouped Split

I compare two validation designs using the same Logistic Regression feature set.

**Random row split:** webpages are randomly assigned to training and testing. Pages from the same client may appear in both sets.

**Client-grouped split:** entire clients are assigned either to training or testing, with zero client overlap.

The grouped split is the more realistic evaluation for this project because pages from the same client may share content strategy, audience, measurement patterns, and other hidden characteristics. The harder question is whether the model generalizes to clients it has never seen.

In [2]:
import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "aliakhtar1010/search-ranking-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Same target used in Week 5
df["declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

# EXACT same 22 features used in Week 5
FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[FEATURES].copy()
y = df["declining_label"].copy()
groups = df["client_id"].copy()

print("Dataset shape:", df.shape)
print("Features:", len(FEATURES))
print("Declining pages:", y.sum())
print("Base decline rate:", round(y.mean(), 3))
print("Missing features:", [c for c in FEATURES if c not in df.columns])

Dataset shape: (30000, 45)
Features: 22
Declining pages: 16262
Base decline rate: 0.542
Missing features: []


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

RANDOM_STATE = 42

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    return labels[order[:k]].mean()


def make_logistic_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ])


# -------------------------
# RANDOM ROW SPLIT
# -------------------------

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

random_model = make_logistic_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]


# -------------------------
# CLIENT-GROUPED SPLIT
# -------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

group_train_idx, group_test_idx = next(
    group_splitter.split(X, y, groups=df["client_id"])
)

X_train_group = X.iloc[group_train_idx]
X_test_group = X.iloc[group_test_idx]

y_train_group = y.iloc[group_train_idx]
y_test_group = y.iloc[group_test_idx]

group_model = make_logistic_model()

group_model.fit(
    X_train_group,
    y_train_group
)

group_scores = group_model.predict_proba(
    X_test_group
)[:, 1]


# -------------------------
# CHECK CLIENT OVERLAP
# -------------------------

train_clients = set(df.iloc[group_train_idx]["client_id"])
test_clients = set(df.iloc[group_test_idx]["client_id"])

client_overlap = len(
    train_clients.intersection(test_clients)
)


# -------------------------
# COMPARISON
# -------------------------

validation_results = pd.DataFrame([
    {
        "split": "Random row split",
        "base_rate": y_test_random.mean(),
        "Precision@10": precision_at_k(random_scores, y_test_random, 10),
        "Precision@20": precision_at_k(random_scores, y_test_random, 20),
        "Precision@50": precision_at_k(random_scores, y_test_random, 50),
    },
    {
        "split": "Client-grouped split",
        "base_rate": y_test_group.mean(),
        "Precision@10": precision_at_k(group_scores, y_test_group, 10),
        "Precision@20": precision_at_k(group_scores, y_test_group, 20),
        "Precision@50": precision_at_k(group_scores, y_test_group, 50),
    }
])

print("Grouped split client overlap:", client_overlap)

validation_results

Grouped split client overlap: 0


,split,base_rate,Precision@10,Precision@20,Precision@50
0,Random row split,0.542000,1.0,0.95,0.82
1,Client-grouped split,0.510952,0.6,0.75,0.72


### Validation Interpretation

The random row split produced stronger ranking results than the client-grouped split:

- Random split: Precision@10 = 1.00, Precision@20 = 0.95, Precision@50 = 0.82
- Client-grouped split: Precision@10 = 0.60, Precision@20 = 0.75, Precision@50 = 0.72

The grouped split also confirmed **zero client overlap** between training and testing.

The gap suggests that randomly mixing webpages from the same clients across training and testing produces a more optimistic estimate of model performance. Pages belonging to one client may share hidden characteristics such as content strategy, audience, measurement patterns, or topic mix.

For this project, I therefore treat the client-grouped result as the more credible validation result because it measures performance on clients the model did not see during training.

The model still performs above the held-out base rate of approximately 0.51 at Precision@20 and Precision@50, but the grouped result is weaker than the random-split result and should be used for public claims.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit

I audited the final feature set using three leakage categories:

1. **Label-derived leakage:** columns used to define `declining_label` must not appear as model features.
2. **Future-window leakage:** features must be available before the outcome being predicted.
3. **Decision-derived leakage:** existing product flags or scores must not be used as model inputs.

The final Week-5 feature set excludes `trend_direction`, `trend_pct`, the constructed decline label, and direct last-30-day versus previous-30-day fields that could reconstruct the trend outcome.

Client IDs are used only for grouping and splitting, never as predictive features.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
LEAKY_OR_EXCLUDED = [
    "trend_direction",
    "trend_pct",
    "declining_label",
    "is_declining_label",

    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",

    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",

    "client_id",
    "content_id"
]

feature_leakage_check = pd.DataFrame({
    "column": LEAKY_OR_EXCLUDED,
    "used_as_feature": [
        col in FEATURES
        for col in LEAKY_OR_EXCLUDED
    ]
})

feature_leakage_check

,column,used_as_feature
0,trend_direction,False
1,trend_pct,False
2,declining_label,False
3,is_declining_label,False
4,impressions_last_30d,False
5,clicks_last_30d,False
6,sessions_last_30d,False
7,impressions_prev_30d,False
8,clicks_prev_30d,False
9,sessions_prev_30d,False


In [5]:
# Deliberate leakage sanity check:
# trend_pct is directly related to the trend label,
# so adding it SHOULD produce suspiciously strong performance.

X_leaky = X.copy()
X_leaky["trend_pct"] = df["trend_pct"]

X_train_leaky = X_leaky.iloc[group_train_idx]
X_test_leaky = X_leaky.iloc[group_test_idx]

leaky_model = make_logistic_model()

leaky_model.fit(
    X_train_leaky,
    y_train_group
)

leaky_scores = leaky_model.predict_proba(
    X_test_leaky
)[:, 1]

leak_test = pd.DataFrame([
    {
        "feature_set": "Honest features",
        "Precision@50": precision_at_k(
            group_scores,
            y_test_group,
            50
        )
    },
    {
        "feature_set": "With leaked trend_pct",
        "Precision@50": precision_at_k(
            leaky_scores,
            y_test_group,
            50
        )
    }
])

leak_test

,feature_set,Precision@50
0,Honest features,0.72
1,With leaked trend_pct,1.00


### Leakage Audit Result

None of the identified label-derived, outcome-window, or identifier columns are present in the final model feature set.

The deliberate leakage experiment confirms that the validation setup is capable of detecting target leakage. When `trend_pct`, a field directly related to the construction of the decline label, was intentionally added to the features, Precision@50 increased from 0.72 to 1.00.

That perfect result is not evidence of a better model. It shows that the model was given information that effectively reveals the answer.

I therefore remove `trend_pct` after the test and retain the original honest feature set. The reported model result remains the client-grouped Precision@50 of 0.72.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim Rewrite

**Too strong:**

> Logistic Regression identifies which webpages need refreshing and substantially outperforms the rule baseline.

**Rewritten claim:**

> On the client-grouped holdout used in this analysis, Logistic Regression ranked pages associated with the decline proxy more effectively near the top of the review queue than the frozen Week-4 rule baseline.

The result is an observed validation result on this dataset, not evidence that the model identifies every page that truly needs a refresh or that refreshing a recommended page will cause future search performance to improve.

The model should therefore be used as **decision support for prioritizing human review**, not as an automatic content-refresh decision.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claim_audit = {
    "random_split_precision_at_50": float(
        precision_at_k(random_scores, y_test_random, 50)
    ),
    "grouped_split_precision_at_50": float(
        precision_at_k(group_scores, y_test_group, 50)
    ),
    "grouped_test_base_rate": float(y_test_group.mean()),
    "client_overlap": int(client_overlap),
    "leaked_precision_at_50": float(
        precision_at_k(leaky_scores, y_test_group, 50)
    ),
    "reported_precision_at_50": float(
        precision_at_k(group_scores, y_test_group, 50)
    )
}

claim_audit

{'random_split_precision_at_50': 0.82,
 'grouped_split_precision_at_50': 0.72,
 'grouped_test_base_rate': 0.5109524582184002,
 'client_overlap': 0,
 'leaked_precision_at_50': 1.0,
 'reported_precision_at_50': 0.72}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.